In [ ]:
from cosipy.nonimaging.bgo.ACSPrepareL2 import ACSPrepareL2
import numpy as np



In [ ]:

events_file = "/data/grb_eliza/dc3_grbs/10-100a/events.csv"
acs_lc_file = "/data/grb_eliza/dc3_grbs/10-100a/ACS_bkg_flux/combined_acs_data.hdf5"
acs_output_path = "/data/grb_eliza/dc3_grbs/10-100a/acs_fits"
start_ori_time = 1835487300.0
detector_list = ["x0", "x1", "y0", "y1", "z0", "z1"]

In [ ]:
acs_prepare_l2 = ACSPrepareL2()

In [ ]:
events_df = acs_prepare_l2.read_event_list_csv(events_file)

In [ ]:
events_df

In [ ]:
acs_lc_list = acs_prepare_l2.raed_acs_lc_from_hdf5(acs_lc_file,events_df,start_ori_time,detector_list)

In [ ]:

for lc in acs_lc_list:

    if lc["counts"] is None:
        continue

    time_rel = lc["time"] - lc["time_start"]

    counts_rebinned = {}

    for det in detector_list:
        t_reb, c_reb, dt_reb, edges_reb, n_samples = acs_prepare_l2.rebin_lightcurve(
            time_rel,
            lc["counts"][det]
        )
        counts_rebinned[det] = c_reb

    lc["time_reb"] = t_reb
    lc["dt_reb"] = dt_reb
    lc["edges_reb"] = edges_reb
    lc["n_samples_reb"] = n_samples
    lc["counts_reb"] = counts_rebinned

In [ ]:
acs_prepare_l2.plot_lc(acs_lc_list[0],'z1')

In [ ]:
sanity_checks = True

for lc in acs_lc_list:
    check = acs_prepare_l2.sanity_checks(lc, detector_list, tol=1e-6)
    if check is False:
        sanity_checks = False

if not sanity_checks:
    print("Issue in the rebinning")
else:
    print("Sanity checks passed")

In [ ]:
for lc in acs_lc_list:
    acs_prepare_l2.create_acs_lc_fits(lc, acs_output_path)

In [ ]:
acs_prepare_l2.plot_acs_lc_from_fits("/data/grb_eliza/dc3_grbs/10-100a/acs_fits/bn171108656.fits",plot_counts=False,save=False)